# Laplace diffusion FRET demo (updated)

This notebook solves a 1D Laplace diffusion equation with distance-dependent FRET,
ported from the original nbformat 3 / Python 2 version to nbformat 4 / Python 3.


In [ ]:
%matplotlib inline

import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt


In [ ]:
def evolve(u, d, fd, idr2, dt):
    """
    Evaluate derivatives of the Laplacian part of the Haas-Steinberg equation and
    update the excited state population `u` at a later time-step.

    Parameters
    ----------
    u : np.ndarray
        Distribution at current time step (modified in place).
    d : float
        Mutual diffusion coefficient.
    fd : np.ndarray
        Deactivation rate by FRET and S1->S0 transition.
    idr2 : float
        Inverse squared grid spacing (currently unused in the update).
    dt : float
        Time-step.
    """
    u[1:-1] += (d * (u[2:] - 2.0 * u[1:-1] + u[0:-2]) - fd * u[1:-1]) * dt
    u[0] = 0.0
    u[-1] = 0.0


def get_initial_distribution(mean, sigma, n_points=100, verbose=False):
    """
    Initialize the distance grid `rda` and probabilities `p` with a uniform distribution
    around a given mean and width.
    """
    n_points = int(n_points) + 2
    a = float(0.5 * (mean * 2.0 - sigma / np.sqrt(1.0 / 12.0)))
    b = float(mean * 2.0 - a)
    if verbose:
        print(f"lower bound: {a}")
        print(f"upper bound: {b}")
        print(f"n points   : {n_points}")
    rda = np.linspace(a, b, n_points)
    p = np.ones_like(rda, dtype=float)
    p /= p.sum()
    p[0], p[-1] = 0.0, 0.0
    return rda, p


def integrate_fret(
    tau0,
    rda_mean,
    sigma,
    rda_n_step,
    forster_radius,
    diffusion_coefficient,
    simulation_time=5.0,
    verbose=False,
    dt=None,
):
    """
    Integrate the 1D diffusion-FRET equation in time using an explicit scheme.

    All distances are in Angstrom and times in ns.
    """
    rda, u = get_initial_distribution(rda_mean, sigma, n_points=rda_n_step, verbose=verbose)

    d = float(diffusion_coefficient)
    fd = (1.0 / tau0) * (1.0 + (forster_radius / rda[1:-1]) ** 6)
    max_rate = max(d, float(fd.max()))
    idr2 = 1.0 / (rda[1] - rda[0])
    if dt is None:
        dt = 1.0 / (2.0 * 20.0 * max_rate)

    time_steps = int(simulation_time / dt)

    if verbose:
        print(f"Diffusion coefficient [A2/ns]: {d}")
        print(f"time-step: {dt}")
        print(f"idr2: {idr2}")

    pt = np.zeros((time_steps, u.shape[0]), dtype=float)
    pt[0, :] = u
    for m in range(0, time_steps - 1):
        pt[m, :] = u
        evolve(u, d, fd, idr2, dt)
    time_axis = np.arange(0, time_steps, dtype=float) * dt
    return time_axis, rda, pt


In [ ]:
time_axis, rda, pt = integrate_fret(
    rda_n_step=100,
    rda_mean=45.0,
    sigma=6.0,
    tau0=4.0,
    forster_radius=50.0,
    diffusion_coefficient=10.0,
    simulation_time=30.0,
    verbose=False,
)

# Molecules that are in the S1-state (unnormalized)
for i in range(0, time_axis.shape[0], 300):
    plt.plot(rda, pt[i])
plt.xlabel('Distance (A)')
plt.ylabel('Population')
plt.title('S1 population vs distance (unnormalized)')
plt.show()

# Molecules that are in the S1-state / normalized
for i in range(0, time_axis.shape[0], 300):
    p_norm = pt[i] / pt[i].sum()
    plt.plot(rda, p_norm)
plt.xlabel('Distance (A)')
plt.ylabel('Probability')
plt.title('Normalized S1 distribution vs distance')
plt.show()

# Fraction of initial concentration
fraction = pt.sum(axis=1)
plt.plot(time_axis, fraction)
plt.xlabel('Time (ns)')
plt.ylabel('Fraction of initial population')
plt.title('Total excited-state population vs time')
plt.show()

# The mean position of the peak over time
times_of_calc_mean = []
mean_distances = []
for i in range(0, time_axis.shape[0], 300):
    p = pt[i, :]
    p = p / p.sum()
    rda_mean_t = float(np.dot(rda, p))
    times_of_calc_mean.append(time_axis[i])
    mean_distances.append(rda_mean_t)

plt.plot(times_of_calc_mean, mean_distances)
plt.xlabel('Time (ns)')
plt.ylabel('Mean distance (A)')
plt.title('Mean donor-acceptor distance vs time')
plt.show()

# Expected distribution of rates (integrated over time)
p_expected = pt.sum(axis=0)
plt.plot(rda, p_expected)
plt.xlabel('Distance (A)')
plt.ylabel('Integrated population')
plt.title('Expected distance distribution')
plt.show()

mean_distance = float(np.dot(rda, p_expected / p_expected.sum()))
print(f'mean distance: {mean_distance}')


In [ ]:
time_axis, rda, pt_w = integrate_fret(
    rda_n_step=100,
    rda_mean=45.0,
    sigma=6.0,
    tau0=4.0,
    forster_radius=50.0,
    diffusion_coefficient=10.0,
    simulation_time=30.0,
    verbose=False,
    dt=0.0025,
)
print(time_axis[1] - time_axis[0])
print(time_axis.shape)

time_axis, rda, pt_o = integrate_fret(
    rda_n_step=100,
    rda_mean=45.0,
    sigma=6.0,
    tau0=4.0,
    forster_radius=50.0,
    diffusion_coefficient=0.1,
    simulation_time=30.0,
    verbose=False,
    dt=0.0025,
)
print(time_axis[1] - time_axis[0])
print(time_axis.shape)
